# Тема 8. FitCoach — фітнес-тренер — домашня робота
**Курс:** AI Fundamentals · Product Management

⚡ Запускай комірки по порядку зверху вниз (Shift+Enter)

## 0. Налаштування

In [1]:
!pip install -q \
    "langchain==1.3.11" \
    "langchain-openai==1.3.3" \
    "langchain-community==0.4.2" \
    "langgraph==1.2.7" \
    "langchainhub==0.1.21" \
    "python-dotenv==1.2.2" \
    "wikipedia==1.4.0" \
    "numexpr==2.14.1" \
    "numpy<3" 2>&1 | grep -v "dependency conflicts"

In [2]:
import os
from datetime import datetime
from typing import List, Dict, Any
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

print("✅ Імпорти виконано.")

✅ Імпорти виконано.


In [3]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ Ключ з Colab Secrets")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    print("✅ Ключ встановлено")

✅ Ключ з Colab Secrets


## 1. Бази даних

In [4]:
def calculate_bmr(weight_kg: float, height_cm: float, age: int, gender: str) -> float:
    """Формула Міффліна-Сан Жеора для BMR."""
    if gender.lower() in ["ч", "чоловік", "m", "male", "чол"]:
        return 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        return 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

# Ключі – стандартні англійські назви рівнів активності.
# "factor" – коефіцієнт для TDEE, "label" – назва для людини,
# "aliases" – основи слів, за якими інструмент впізнає рівень у тексті в будь-якій формі
# ("помірн" знайде і "помірна", і "помірною").
ACTIVITY_LEVELS = {
    "sedentary":   {"factor": 1.2,   "label": "мінімальна",   "aliases": ["мінімальн", "сидяч"]},
    "light":       {"factor": 1.375, "label": "легка",        "aliases": ["легк"]},
    "moderate":    {"factor": 1.55,  "label": "помірна",      "aliases": ["помірн", "середн"]},
    "active":      {"factor": 1.725, "label": "висока",       "aliases": ["висок"]},
    "very_active": {"factor": 1.9,   "label": "екстремальна", "aliases": ["екстремальн"]},
}

# Ключі груп англійською. "label" – назва для людини,
# "aliases" – основи слів, за якими інструмент впізнає групу в будь-якому відмінку
# ("спин" знайде і "спина", і "спину", і "спині").
# contraindications: стани, за яких вправу не пропонуємо (значення з ДЗ2).
EXERCISES_DB = {
    "chest": {
        "label": "груди",
        "aliases": ["груд"],
        "home": [
            {"name": "Віджимання від стіни", "level": "beginner",
             "technique": "Стань на крок від стіни, долоні на рівні плечей, повільно згинай і розгинай руки",
             "sets": "2-3", "reps": "10-12", "contraindications": []},
            {"name": "Віджимання від підлоги з колін", "level": "beginner",
             "technique": "Коліна на підлозі, тіло рівне від колін до голови, опускай груди до підлоги",
             "sets": "2-3", "reps": "8-10", "contraindications": ["acute_back_injury"]},
            {"name": "Віджимання від підлоги", "level": "intermediate",
             "technique": "Руки на ширині плечей, тіло пряме, опускайся до торкання грудьми підлоги",
             "sets": "3", "reps": "10-15", "contraindications": ["acute_back_injury"]},
        ],
        "gym": [
            {"name": "Жим штанги лежачи", "level": "intermediate",
             "technique": "Лопатки зведені, стопи на підлозі, опускай штангу до грудей",
             "sets": "4", "reps": "8-12", "contraindications": ["acute_back_injury"]},
        ],
    },
    "back": {
        "label": "спина",
        "aliases": ["спин", "попере"],
        "home": [
            {"name": "Котик-корова", "level": "beginner",
             "technique": "Стоячи на четвереньках, по черзі плавно прогинай і округлюй спину",
             "sets": "2", "reps": "10", "contraindications": ["acute_back_injury"]},
            {"name": "Сідничний міст", "level": "beginner",
             "technique": "Лежачи на спині, ноги зігнуті, плавно підіймай таз і опускай",
             "sets": "2-3", "reps": "10-12", "contraindications": ["acute_back_injury"]},
            {"name": "Птах-собака", "level": "beginner",
             "technique": "На четвереньках витягни протилежні руку й ногу, затримайся на 3 секунди, поміняй сторону",
             "sets": "2", "reps": "8 на кожну сторону", "contraindications": ["acute_back_injury"]},
            {"name": "Супермен", "level": "intermediate",
             "technique": "Лежачи на животі, одночасно підійми руки й ноги, затримайся на 2-3 секунди",
             "sets": "3", "reps": "10", "contraindications": ["lower_back_pain", "acute_back_injury"]},
        ],
        "gym": [
            {"name": "Підтягування", "level": "intermediate",
             "technique": "Хват трохи ширше плечей, підтягуйся до підборіддя вище перекладини",
             "sets": "3-4", "reps": "максимум", "contraindications": ["acute_back_injury"]},
            {"name": "Станова тяга зі штангою", "level": "intermediate",
             "technique": "Підйом штанги з підлоги з рівною спиною, рух за рахунок ніг і тазу",
             "sets": "3", "reps": "8", "contraindications": ["lower_back_pain", "acute_back_injury"]},
        ],
    },
    "legs": {
        "label": "ноги",
        "aliases": ["ног", "ніг", "стегн", "сідниц"],
        "home": [
            {"name": "Присідання на стілець", "level": "beginner",
             "technique": "Повільно сідай на стілець і вставай без допомоги рук, спина пряма",
             "sets": "2-3", "reps": "10", "contraindications": []},
            {"name": "Присідання", "level": "beginner",
             "technique": "Стопи на ширині плечей, спина пряма, присідай до паралелі стегон з підлогою",
             "sets": "3", "reps": "12-15", "contraindications": ["acute_back_injury"]},
            {"name": "Випади", "level": "intermediate",
             "technique": "Крок уперед, заднє коліно майже торкається підлоги, спина пряма",
             "sets": "3", "reps": "10 на кожну ногу", "contraindications": ["acute_back_injury"]},
        ],
    },
    "core": {
        "label": "прес",
        "aliases": ["прес", "живіт", "живот"],
        "home": [
            {"name": "Мертвий жук", "level": "beginner",
             "technique": "Лежачи на спині, руки й ноги вгору, по черзі опускай протилежні руку й ногу, поперек притиснутий",
             "sets": "2", "reps": "8 на кожну сторону", "contraindications": ["acute_back_injury"]},
            {"name": "Планка з колін", "level": "beginner",
             "technique": "Упор на передпліччя й коліна, тіло рівне, тримай 15-20 секунд",
             "sets": "3", "reps": "15-20 секунд", "contraindications": ["acute_back_injury"]},
            {"name": "Скручування", "level": "intermediate",
             "technique": "Лежачи на спині, ноги зігнуті, підіймай плечі від підлоги",
             "sets": "3", "reps": "15", "contraindications": ["lower_back_pain", "neck_pain", "acute_back_injury"]},
        ],
    },
    "neck_shoulders": {
        "label": "шия і плечі",
        "aliases": ["шия", "шию", "шиї", "шиєю", "плеч", "лопат"],
        "home": [
            {"name": "Зведення лопаток сидячи", "level": "beginner",
             "technique": "Сидячи рівно, зведи лопатки разом, затримайся на 3 секунди, розслаб",
             "sets": "2", "reps": "10", "contraindications": []},
            {"name": "Кругові рухи плечима", "level": "beginner",
             "technique": "Повільно обертай плечима назад, потім уперед",
             "sets": "2", "reps": "10 у кожен бік", "contraindications": []},
            {"name": "Нахили голови до плеча", "level": "beginner",
             "technique": "Повільно нахили голову до плеча, утримуй 15 секунд, без ривків",
             "sets": "2", "reps": "по 15 секунд у кожен бік", "contraindications": ["acute_back_injury"]},
        ],
    },
}

home_count = sum(len(g.get("home", [])) for g in EXERCISES_DB.values())
gym_count = sum(len(g.get("gym", [])) for g in EXERCISES_DB.values())
print(f"✅ База: {len(EXERCISES_DB)} груп м'язів, {home_count} вправ для дому, "
      f"{gym_count} для залу, {len(ACTIVITY_LEVELS)} рівнів активності")


✅ База: 5 груп м'язів, 16 вправ для дому, 3 для залу, 5 рівнів активності


### ✍️ Продуктовий коментар

**Чому база локальна, а не з фітнес-API.** Для навчального прототипу локальна база дає повний контроль: я сама вирішую, які вправи є, як вони описані і для кого небезпечні. Для FitCoach це головне, бо безпека людини з болем у спині залежить від розмітки протипоказань, а зовнішній API таких полів може не мати. Плюс немає залежності від чужого сервісу й додаткових витрат.

**Які обмеження це створює.** 16 вправ для дому – це мало для реального продукту: план швидко стане одноманітним. Базу треба оновлювати вручну. Розмітку протипоказань я зробила обережно, але я не фізіотерапевт: у реальному продукті її має перевірити фахівець.

**Що відбудеться, якщо вправи немає в базі.** У «знімку до» на шаблоні користувачка написала «вправи на **спину**», а шаблон шукав точне слово «спина» і не знайшов групу. Агент 6 разів викликав інструмент з тими самими аргументами, а потім вигадав вправи сам, серед них «супермен» і нахили вперед, ризиковані при болю в попереку. Це неприйнятно: невдача пошуку не повинна перетворюватися на непровірені поради. Тепер інструмент на невідому групу чесно відповідає «групу не розпізнано» й називає доступні. Але як агент поводиться далі, залежить від запуску: на запит «вправи на руки» він в одному запуску тихо підмінив «руки» на «плечі», в іншому чесно сказав, що таких вправ немає, у третьому взагалі не викликав інструмент (див. тести 14 і 15).

**Що я змінила в базі:**
- 5 груп замість 3, у кожній 3–4 вправи для дому. У групах «груди», «ноги» і «шия і плечі» є вправи без протипоказань. У «спині» й «пресі» всі вправи мають протипоказання (щонайменше гостра травма спини), тож якщо в групі не лишилося безпечних вправ, інструмент пропонує найм'якші вправи без протипоказань з інших груп;
- кожна вправа має поле `contraindications`: за ним інструмент відсікає небезпечне для людини з болем у спині чи шиї;
- кожна група має `aliases` (основи слів), щоб пошук працював у будь-якому відмінку: «спину», «попереку»;
- рівні активності теж переписала з англійськими ключами й основами слів, щоб «помірною» чи «сидяча робота» впізнавалися так само, як «помірна»;
- підтягування перенесла в зал (потрібна перекладина), а «обернені віджимання від стільця» прибрала, бо це вправа на трицепс, а не на спину.

## 2. Інструменти (@tool)

In [5]:
# ============================================================
# Допоміжні функції: числа в тексті
# ============================================================
def find_number(text: str, stems: list[str]) -> float | None:
    """Шукає число одразу після слова з однією з основ ("ваг" -> "вага 72", "вагою 72").
    Знак мінус зберігається, щоб перевірка діапазону помітила помилку ("вага -72")."""
    for stem in stems:
        match = re.search(stem + r"\w*\D{0,10}?(-?\d+(?:[.,]\d+)?)", text)
        if match:
            return float(match.group(1).replace(",", "."))
    return None


def all_numbers(text: str) -> list[float]:
    """Усі числа в тексті, разом зі знаком мінус."""
    return [float(n.replace(",", ".")) for n in re.findall(r"-?\d+(?:[.,]\d+)?", text)]


# ============================================================
# Інструмент 1: ІМТ
# ============================================================
@tool
def bmi_calculator(query: str) -> str:
    """Розраховує індекс маси тіла (ІМТ) і називає категорію за класифікацією ВООЗ.
    Використовуй, коли користувач питає свій ІМТ або чи його вага в межах норми.
    Підходить лише для дорослих від 18 років: для дітей і підлітків категорії інші.
    Передавай лише ті значення, які людина назвала сама (зараз або раніше в діалозі).
    Формат запиту: 'вага <кг>, зріст <см>'"""
    if not query.strip():
        return "Щоб порахувати ІМТ, мені потрібні вага (кг) і зріст (см)."

    q = query.lower()
    weight_kg = find_number(q, ["ваг", "важ"])
    height_cm = find_number(q, ["зріст", "зрост"])

    # Якщо слів "вага" і "зріст" немає, пробуємо обидва порядки чисел
    if weight_kg is None or height_cm is None:
        numbers = all_numbers(q)
        if len(numbers) < 2:
            missing = "зріст (см)" if weight_kg is not None or len(numbers) == 1 else "вагу (кг) і зріст (см)"
            return f"Щоб порахувати ІМТ, напиши ще {missing}."
        a, b = numbers[0], numbers[1]
        options = [(w, h) for w, h in [(a, b), (b, a)] if 20 <= w <= 300 and 100 <= h <= 250]
        if len(options) == 1:
            weight_kg, height_cm = options[0]
        elif len(options) == 2:
            return f"Уточни, будь ласка: {a:g} – це вага чи зріст?"
        else:
            weight_kg, height_cm = a, b  # жоден порядок не реальний, далі спрацює перевірка діапазону

    if not 20 <= weight_kg <= 300:
        return f"Вага {weight_kg:g} кг виглядає незвично. Перевір, будь ласка, число."
    if not 100 <= height_cm <= 250:
        return f"Зріст {height_cm:g} см виглядає незвично. Перевір, будь ласка, число."

    bmi = weight_kg / (height_cm / 100) ** 2

    # Класифікація ВООЗ. Порада про лікаря лише для крайніх значень і як турбота, а не вирок
    if bmi < 16:
        category = "виражена нестача маси тіла"
        note = "У такому випадку варто порадитися з лікарем, щоб тренування були безпечними."
    elif bmi < 18.5:
        category = "нестача маси тіла"
        note = ""
    elif bmi < 25:
        category = "нормальна маса тіла"
        note = ""
    elif bmi < 30:
        category = "надлишкова маса тіла"
        note = ""
    else:
        category = "ожиріння"
        note = "Лікар може допомогти скласти план, комфортний саме для тебе."

    return (
        f"ІМТ: {bmi:.1f} (категорія за ВООЗ: {category}).\n"
        + (f"{note}\n" if note else "")
        + "ІМТ – лише одна цифра: він не враховує м'язи й не описує ні здоров'я, ні силу.\n"
        "Якщо хочеш, можу розрахувати норму калорій або підібрати вправи."
    )


# ============================================================
# Інструмент 2: калорії
# ============================================================
# Безпечна нижня межа калорій при схудненні
CALORIE_FLOOR = {"female": 1200, "male": 1500}

# Цілі як відсоток від норми підтримки (ілюстративно, у продукті має затвердити дієтолог)
GOALS = {
    "gentle_loss": {"label": "лагідне схуднення", "change": -0.15},
    "fast_loss":   {"label": "швидке схуднення",  "change": -0.25},
    "maintain":    {"label": "підтримка ваги",    "change": 0.0},
    "gain":        {"label": "набір маси",        "change": 0.10},
}


def calorie_targets(tdee: float, floor: int) -> dict:
    """Лише математика: норма для кожної цілі. Схуднення не опускається нижче безпечної межі."""
    targets = {}
    for key, goal in GOALS.items():
        kcal = tdee * (1 + goal["change"])
        floored = goal["change"] < 0 and kcal < floor
        targets[key] = (max(kcal, floor) if goal["change"] < 0 else kcal, floored)
    return targets


def detect_goal(q: str) -> str | None:
    """Мета з тексту. Швидке схуднення лише тоді, коли людина прямо просить швидко."""
    if any(s in q for s in ["схуд", "худн", "скинут", "зниз"]):
        return "fast_loss" if "швидк" in q else "gentle_loss"
    if any(s in q for s in ["набр", "набор", "набир", "м'яз", "мʼяз", "м’яз"]):
        return "gain"
    if any(s in q for s in ["підтрим", "зберег", "утрим"]):
        return "maintain"
    return None


def describe_targets(goal: str | None, bmi: float, targets: dict, floor: int, weight_kg: float) -> str:
    """Політика показу: що бачить людина залежно від її мети та ІМТ."""
    def line(key):
        kcal, floored = targets[key]
        extra = f" (не нижче безпечної межі {floor} ккал)" if floored else ""
        return f"{GOALS[key]['label'].capitalize()}: {kcal:.0f} ккал/день{extra}"

    maintain = line("maintain")

    if goal is None:
        return maintain + "\nЯка в тебе мета: підтримати вагу, схуднути чи набрати масу? Тоді підкажу точніше."
    if goal == "maintain":
        return maintain

    if goal in ("gentle_loss", "fast_loss"):
        if bmi < 18.5:
            return (maintain + "\nЗараз твоя вага нижча за норму, тому дефіцит калорій я не рахую. "
                    "Якщо хочеш змінити вагу, краще спершу порадитися з лікарем.")
        if goal == "gentle_loss":
            return maintain + "\n" + line("gentle_loss")
        return (maintain + "\n" + line("fast_loss") +
                "\nЧесно про ризики: швидкий темп частіше забирає м'язи, дає втому й зриви. "
                f"Лагідний темп надійніший: {targets['gentle_loss'][0]:.0f} ккал/день.")

    # goal == "gain"
    if bmi < 18.5:
        doctor = "\nПри такій низькій вазі варто порадитися з лікарем." if bmi < 16 else ""
        return maintain + "\n" + line("gain") + doctor
    if bmi < 30:
        return (maintain + "\n" + line("gain") +
                "\nЩоб приріст був переважно м'язами, додай силові тренування й достатньо білка "
                f"(близько {weight_kg * 1.6:.0f}–{weight_kg * 2.2:.0f} г/день).")
    return (maintain + "\nНабір маси я не рахую: краще спершу порадитися з ендокринологом, "
            "він допоможе зрозуміти, що саме потрібно твоєму організму.")


@tool
def calorie_calculator(query: str) -> str:
    """Розраховує денну норму калорій за формулою Міффліна-Сан Жеора і підказує норму для мети людини.
    Використовуй, коли користувач питає, скільки калорій йому потрібно на день,
    як їсти, щоб підтримати вагу, схуднути чи набрати масу.
    Підходить лише для дорослих від 18 років.
    Потрібні: вага, зріст, вік, стать, рівень активності. Мета необов'язкова.
    Передавай лише те, що людина сказала сама (зараз або раніше в діалозі).
    Якщо стать, рівень активності чи мету людина не називала, не додавай їх: інструмент сам попросить уточнити.
    Швидке схуднення передавай лише тоді, коли людина прямо просить швидко.
    Формат запиту: 'вага <кг>, зріст <см>, <вік> років, <стать>, <рівень активності>, мета: <мета>'
    Стать: жінка або чоловік. Рівень активності: мінімальна, легка, помірна, висока або екстремальна.
    Мета: схуднути, швидко схуднути, підтримати вагу або набрати масу."""
    if not query.strip():
        return "Щоб порахувати норму калорій, мені потрібні вага, зріст, вік, стать і рівень активності."

    q = query.lower()
    weight_kg = find_number(q, ["ваг", "важ"])
    height_cm = find_number(q, ["зріст", "зрост"])
    age_match = re.search(r"(-?\d+)\s*(?:рік|роки|років|р\.)", q) or re.search(r"вік\D{0,5}?(-?\d+)", q)
    age = int(age_match.group(1)) if age_match else None

    # Якщо підписів немає, беремо перші три числа за порядком: вага, зріст, вік
    if weight_kg is None or height_cm is None or age is None:
        numbers = all_numbers(q)
        if len(numbers) < 3:
            return "Щоб порахувати норму калорій, напиши вагу (кг), зріст (см) і вік."
        weight_kg, height_cm, age = numbers[0], numbers[1], int(numbers[2])

    if not 20 <= weight_kg <= 300:
        return f"Вага {weight_kg:g} кг виглядає незвично. Перевір, будь ласка, число."
    if not 100 <= height_cm <= 250:
        return f"Зріст {height_cm:g} см виглядає незвично. Перевір, будь ласка, число."
    if 0 < age < 18:
        return "Норму калорій FitCoach розраховує лише для дорослих від 18 років."
    if not 18 <= age <= 100:
        return f"Вік {age} виглядає незвично. Перевір, будь ласка, число."

    # Стать: спершу жіночі слова, бо "female" містить "male"
    if any(w in q for w in ["жін", "female"]):
        sex = "female"
    elif any(w in q for w in ["чолов", "male"]):
        sex = "male"
    else:
        sex = None

    activity = None
    for key, level in ACTIVITY_LEVELS.items():
        if any(alias in q for alias in level["aliases"]):
            activity = key
            break

    # Не вгадуємо: стать і активність помітно змінюють результат
    missing = []
    if sex is None:
        missing.append("стать (формула для жінок і чоловіків різна)")
    if activity is None:
        levels = ", ".join(level["label"] for level in ACTIVITY_LEVELS.values())
        missing.append(f"рівень активності ({levels})")
    if missing:
        return "Щоб порахувати точно, уточни, будь ласка: " + "; ".join(missing) + "."

    bmr = calculate_bmr(weight_kg, height_cm, age, "чоловік" if sex == "male" else "жінка")
    level = ACTIVITY_LEVELS[activity]
    tdee = bmr * level["factor"]
    bmi = weight_kg / (height_cm / 100) ** 2
    floor = CALORIE_FLOOR[sex]
    targets = calorie_targets(tdee, floor)

    return (
        f"Базовий метаболізм (BMR): {bmr:.0f} ккал/день\n"
        f"Денна норма при активності «{level['label']}» (TDEE): {tdee:.0f} ккал/день\n\n"
        + describe_targets(detect_goal(q), bmi, targets, floor, weight_kg)
    )


# ============================================================
# Інструмент 3: вправи з фільтром протипоказань
# ============================================================
CONDITION_LABELS = {"lower_back_pain": "болю в попереку", "neck_pain": "болю в шиї"}
LOCATION_LABELS = {"home": "вдома", "gym": "у залі"}

# Тривожні ознаки (з ДЗ2): за них вправ не підбираємо, а радимо лікаря.
# Шаблони регулярних виразів: "німі" ловить "німіє", "німіти", "оніміння";
# \b не дає "нічн" спрацювати всередині слова "технічно".
RED_FLAG_PATTERNS = [
    r"віддає\s+[ву]\s+(ног|рук|сідниц)",
    r"німі",
    r"\bнічн",
    r"вночі|уночі|ночами",
]
# Травма – окремо, бо лише її людина може заперечити ("без травм", "травм не було")
INJURY_PATTERNS = [r"травм", r"\bвпа(в|ла)\b", r"падін"]
NO_INJURY_PATTERNS = [r"без\s+травм", r"травм\w*\s+(не\s+було|немає|нема)\b", r"не\s+було\s+травм"]

PAIN_WORDS = ["бол", "біль", "ниє", "ниють", "тягне", "ломить", "скут"]
BACK_WORDS = ["попере", "спин"]
NECK_WORDS = ["шия", "шию", "шиї", "шиєю"]


def has_red_flag(q: str) -> bool:
    """True, якщо в запиті є тривожна ознака. Заперечення травми не вимикає інші ознаки."""
    if any(re.search(p, q) for p in RED_FLAG_PATTERNS):
        return True
    injury = any(re.search(p, q) for p in INJURY_PATTERNS)
    injury_denied = any(re.search(p, q) for p in NO_INJURY_PATTERNS)
    return injury and not injury_denied


def parse_parts(q: str) -> dict:
    """Розбирає запит формату 'група: спина; місце: вдома; біль: поперек' на поля."""
    parts = {}
    for chunk in q.split(";"):
        if ":" in chunk:
            key, value = chunk.split(":", 1)
            parts[key.strip()] = value.strip()
    return parts


def format_exercises(exercises: list) -> list:
    lines = []
    for ex in exercises:
        lines.append(f"\n{ex['name']}")
        lines.append(f"  Як виконувати: {ex['technique']}")
        lines.append(f"  Підходи: {ex['sets']} × {ex['reps']}")
    return lines


@tool
def exercise_lookup(query: str) -> str:
    """Підбирає вправи з бази FitCoach за групою м'язів, місцем (вдома/зал) і станом людини.
    Використовуй, коли людина питає про вправи або що робити при болю чи скутості в спині, попереку, шиї.
    Інструмент сам відсікає вправи, протипоказані при болю, і не підбирає вправ, якщо є тривожні ознаки.
    Передавай лише те, що людина сказала сама (зараз або раніше в діалозі).
    Якщо людина згадувала біль, обов'язково передай його в полі 'біль'; якщо не згадувала, напиши 'біль: немає'.
    Формат запиту: 'група: <група>; місце: <місце>; біль: <де болить або немає>; рівень: <рівень>'
    Групи: спина, груди, ноги, прес, шия і плечі. Місце: вдома або зал. Рівень: початківець або середній."""
    labels = ", ".join(g["label"] for g in EXERCISES_DB.values())
    if not query.strip():
        return f"Вкажи групу м'язів. Доступні: {labels}."

    q = query.lower()
    parts = parse_parts(q)

    # 1) Тривожні ознаки: зупиняємося й дбайливо радимо лікаря
    if has_red_flag(q):
        return ("Біль, що віддає в ногу чи руку, оніміння, біль уночі чи після травми – це ознаки, "
                "які варто спершу показати лікарю. Тому зараз я не підбираю вправи. "
                "Коли лікар дасть добро, я з радістю допоможу почати.")

    # 2) Стан людини: де болить
    pain_text = parts.get("біль", q if any(w in q for w in PAIN_WORDS) else "")
    if pain_text in ("немає", "ні", "-"):
        pain_text = ""
    conditions = set()
    if any(w in pain_text for w in BACK_WORDS):
        conditions.add("lower_back_pain")
    if any(w in pain_text for w in NECK_WORDS):
        conditions.add("neck_pain")

    # 3) Група м'язів; якщо не названа, але болить спина чи шия, беремо цю зону
    group_text = parts.get("група", q)
    group_key = next((key for key, g in EXERCISES_DB.items()
                      if any(alias in group_text for alias in g["aliases"])), None)
    if group_key is None:
        if "lower_back_pain" in conditions:
            group_key = "back"
        elif "neck_pain" in conditions:
            group_key = "neck_shoulders"
        else:
            return f"Групу м'язів не розпізнано. Доступні: {labels}."

    location = "gym" if "зал" in parts.get("місце", q) else "home"
    # При болю пропонуємо лише вправи для початківця
    beginner_only = "початків" in parts.get("рівень", q) or bool(conditions)

    group = EXERCISES_DB[group_key]
    candidates = group.get(location, [])
    if not candidates:
        return f"Вправ для групи «{group['label']}» {LOCATION_LABELS[location]} у базі немає."

    # 4) Фільтр у коді: протипоказання, потім рівень
    safe = [ex for ex in candidates if not conditions & set(ex["contraindications"])]
    suitable = [ex for ex in safe if ex["level"] == "beginner" or not beginner_only]
    reasons = ", ".join(CONDITION_LABELS[c] for c in sorted(conditions))

    # 5) Якщо нічого не лишилося: найм'якші вправи без протипоказань з усієї бази
    if not suitable:
        gentle = [ex for g in EXERCISES_DB.values() for ex in g.get("home", []) if not ex["contraindications"]]
        lines = [f"У групі «{group['label']}» {LOCATION_LABELS[location]} не знайшлося вправ, "
                 f"безпечних для тебе зараз. Ось найм'якші вправи без протипоказань, які можна робити вдома:"]
        return "\n".join(lines + format_exercises(gentle))

    lines = [f"Вправи на групу «{group['label']}» {LOCATION_LABELS[location]}:"]
    lines += format_exercises(suitable)
    # Людині не потрібне число відсіяних вправ: важливо, що її стан враховано
    if len(safe) < len(candidates):
        lines.append(f"\nПідібрано з урахуванням {reasons}: вправи, які можуть його посилити, не показую.")
    if len(suitable) < len(safe):
        lines.append("Вправи середнього рівня додамо пізніше, коли ці стануть легкими.")
    if conditions:
        lines.append("Якщо під час вправи біль посилюється, зупинись.")
    return "\n".join(lines)


tools = [bmi_calculator, calorie_calculator, exercise_lookup]
print("✅ Інструменти FitCoach створено.")

✅ Інструменти FitCoach створено.


In [6]:
# Пряма перевірка інструментів, без моделі: доказ, що працює саме код
red_flag_checks = [
    "група: спина; біль: поперек, німіє нога, без травм",    # заперечення травми не вимикає оніміння
    "група: спина; біль: поперек, почала німіти нога",       # "німіти", а не лише "німіє"
    "група: шия і плечі; біль: шия, віддає в руку",          # біль віддає в руку
    "група: спина; біль: поперек, болить уночі",             # нічний біль
    "група: спина; біль: поперек, травма не проходить",      # травма, яку не заперечили
    "група: спина; біль: поперек, травм не було",            # травму заперечено: вправи можна
]
for q in red_flag_checks:
    print("👤", q)
    print(exercise_lookup.invoke({"query": q}))
    print("-" * 40)

print("👤 вага -72, зріст 165")
print(bmi_calculator.invoke({"query": "вага -72, зріст 165"}))
print("-" * 40)
print("👤 калорії для 16 років")
print(calorie_calculator.invoke({"query": "вага 60, зріст 165, 16 років, жінка, легка активність"}))

👤 група: спина; біль: поперек, німіє нога, без травм
Біль, що віддає в ногу чи руку, оніміння, біль уночі чи після травми – це ознаки, які варто спершу показати лікарю. Тому зараз я не підбираю вправи. Коли лікар дасть добро, я з радістю допоможу почати.
----------------------------------------
👤 група: спина; біль: поперек, почала німіти нога
Біль, що віддає в ногу чи руку, оніміння, біль уночі чи після травми – це ознаки, які варто спершу показати лікарю. Тому зараз я не підбираю вправи. Коли лікар дасть добро, я з радістю допоможу почати.
----------------------------------------
👤 група: шия і плечі; біль: шия, віддає в руку
Біль, що віддає в ногу чи руку, оніміння, біль уночі чи після травми – це ознаки, які варто спершу показати лікарю. Тому зараз я не підбираю вправи. Коли лікар дасть добро, я з радістю допоможу почати.
----------------------------------------
👤 група: спина; біль: поперек, болить уночі
Біль, що віддає в ногу чи руку, оніміння, біль уночі чи після травми – це озн

### ✍️ Продуктовий коментар

**Чому один параметр `query: str`, а не окремі числа.** Так радить курс: з кількома числовими параметрами агент може передати їх неправильно. Окремі параметри я сама не перевіряла. Але побачила зворотний бік рядка: агент сам переформульовує запит. У «знімку до» людина не вказала стать, а агент сам дописав «жінка» в запит до калькулятора. Тому інструменти тепер не вгадують важливе, а повертають «уточни, будь ласка». Для вправ я попросила агента передавати рядок у форматі `група: ...; місце: ...; біль: ...`, тож розбір запиту робить модель, а рішення – код.

**Навіщо перевірка діапазону ваги й зросту.** Без неї бот впевнено порахує абсурдний ІМТ і дасть пораду на його основі. Я перевірила це на практиці: на «165 72» шаблон узяв 165 як вагу. Перевірка діапазону зупинила розрахунок, а тепер інструмент пробує обидва порядки чисел і бере єдиний реальний. Ще одну ваду знайшла під час перевірки: парсер ігнорував мінус, і «вага -72» ставала 72. Тепер знак зберігається, і перевірка діапазону ловить помилку.

**Ризик фрази про спортсменів.** Людина може вирішити «я спортивна, мені ІМТ не стосується» і проігнорувати реальний сигнал. Але фраза потрібна: ІМТ – грубий орієнтир, він не враховує м'язи, стать і вік. Я переформулювала її так, щоб вона знімала тиск цифри, а не давала привід її ігнорувати: «ІМТ – лише одна цифра: він не враховує м'язи й не описує ні здоров'я, ні силу».

**Що я змінила і чому:**
- `bmi_calculator`: прибрала непрохані поради на кшталт «збільш активність, переглянь раціон». Людина питала цифру, а отримувала оцінку. Тепер лише факт за ВООЗ і м'яка пропозиція наступного кроку. Дорослі категорії ІМТ не підходять для дітей і підлітків, але вік в інструмент не передається, тож обмеження «від 18 років» тут поки лише інструкція агенту в описі інструмента, а не перевірка в коді.
- `calorie_calculator`: не вважає людину чоловіком за замовчуванням і не ставить «помірну» активність, а перепитує. Знайшла ваду шаблону: «female» містить «male», тож шаблон рахував англійське «female» як чоловіка. Розрахунок відокремила від політики показу: інструмент показує лише норму для мети людини. Швидке схуднення лише на пряме прохання й з чесним поясненням ризиків, дефіцит не рахується при нестачі маси, набір при ожирінні – порада ендокринолога. Безпечна межа 1200/1500 ккал спрацювала навіть для типової користувачки: жінка 43 років із сидячою роботою. Для людей молодших за 18 років код норму не рахує.
- `exercise_lookup`: тривожні ознаки (біль віддає в ногу чи руку, оніміння, біль уночі, травма) зупиняють підбір і ведуть до лікаря; біль у попереку чи шиї відсікає протипоказані вправи кодом, а не моделлю; відповідь каже, що вправи підібрано з урахуванням болю, але без числа відсіяних: людині важливо, що її стан враховано, а не статистика; якщо безпечних немає, пропонує найм'якші вправи замість порожнечі.

**Що показала пряма перевірка (комірка вище).** Під час перевірки я знайшла три вади в розпізнаванні тривожних ознак: «без травм» вимикало всі ознаки разом (і «німіє нога, без травм» проходило); «почала німіти» і «віддає в руку» не розпізнавалися; ознака «нічн» спрацьовувала всередині слова «технічно». Я виправила це: заперечення тепер стосується лише травми, а «травма не проходить» не вважається запереченням. Пряма перевірка без моделі показує, що код зупиняє всі п'ять тривожних випадків і пропускає «травм не було».

**Обмеження.** Тривожні ознаки код розпізнає за словами, тож інші формулювання (наприклад, «поколює в нозі») він може не впізнати. Падіння й травма перевіряються разом, тому «впала вчора, без травм» код пропускає, хоча падіння з новим болем у спині варто показати лікарю: падіння треба перевіряти окремо від заперечення травми. І код перевіряє лише те, що агент йому передав: якщо агент не передасть ознаку, перевірка не спрацює (див. тест 14).

## 3. Модель

In [7]:
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

test = chat_model.invoke("Одним реченням підтвердь готовність працювати як фітнес-тренер.")
print("✅ Модель ОК:", test.content)

✅ Модель ОК: Я готовий працювати як фітнес-тренер і допомагати людям досягати їхніх цілей у фізичному розвитку.


### ✍️ Продуктовий коментар

**Чому `temperature=0.3`, а не 0.7.** Від температури залежить, наскільки по-різному модель формулює відповідь і обирає наступний крок. Самі розрахунки ІМТ і калорій вона не змінює: числа рахує код інструментів, а модель їх лише переказує. Але низька температура робить стабільнішим те, що залежить від моделі: чи викличе вона інструмент, як сформулює запит до нього, чи не «прикрасить» результат. Для фітнес-бота, де помилка може нашкодити, передбачуваність важливіша за різноманітність формулювань. Навіть з 0.3 поведінка між запусками відрізнялася (див. коментар до тестування), тож повної передбачуваності температура не дає.

**Що означає `max_tokens=512`.** Це стеля довжини однієї відповіді: захист від надто довгих текстів і зайвих витрат. Українські тексти займають більше токенів, ніж англійські, тож я перевірила, чи не обрізаються відповіді: у тестах код позначає відповідь, якщо модель зупинилася через ліміт. Жодна відповідь не обрізалася, навіть списки з трьох вправ із технікою. Але детальний план тренування на кілька днів у цей ліміт, найімовірніше, вже не вмістився б.

**`gpt-4o-mini` чи `gpt-4o`.** Для навчального прототипу я обрала mini: вона в рази дешевша й швидша, а найважливіші рішення (фільтр протипоказань, тривожні ознаки, безпечна межа калорій) ухвалює код. Але тести показали помилки передачі даних в інструменти: модель губила біль і вгадувала стать. Тому її придатність для реального продукту потребує додаткової перевірки. Сильніша модель, можливо, краще впоралася б із нечіткими повідомленнями, і в повній версії FitCoach я б розділила ролі: мала модель розбирає запит, сильніша формує відповідь. Але навіть із сильнішою моделлю стан людини безпечніше витягувати кодом, а не довіряти агенту.

## 4. Агент

In [8]:
SYSTEM_PROMPT = """
Ти — FitCoach, доброзичливий фітнес-асистент для дорослих, які тільки починають займатися вдома.
Серед твоїх користувачів багато людей із сидячою роботою, які відчувають дискомфорт у спині чи шиї
і бояться собі нашкодити.

Твої можливості:
- розрахувати ІМТ (bmi_calculator);
- розрахувати денну норму калорій для мети людини (calorie_calculator);
- підібрати вправи з бази FitCoach з урахуванням болю (exercise_lookup).

Тон:
- відповідай українською, звертайся на «ти», коротко й по суті;
- говори про себе без родових форм: «можу», «не вдалося», «підібрано», а не «міг», «могла», «підібрав»;
- ніколи не оцінюй вагу, тіло чи старанність людини, не називай її ледачою і не дорікай;
- підтримуй м'яко: регулярність важливіша за швидкість, маленький крок — уже успіх.

Правила роботи з інструментами:
- усі числа (ІМТ, калорії) і всі вправи бери лише з результатів інструментів; не вигадуй вправ і не рахуй сам;
- якщо інструмент не знайшов вправ або повернув помилку, чесно скажи про це і запропонуй, що уточнити;
  не підміняй результат власними порадами;
- якщо інструмент просить уточнити дані, постав людині одне питання про все, чого бракує;
- якщо інструмент двічі повернув ту саму помилку, не викликай його втретє — уточни в людини;
- стать, рівень активності й мету не вгадуй: передавай лише те, що людина сказала сама;
- швидке схуднення передавай у calorie_calculator лише тоді, коли людина прямо просить швидко;
- для exercise_lookup передавай запит у форматі 'група: ...; місце: ...; біль: ...; рівень: ...';
- рахуй і підбирай лише те, про що людина просить; не запускай інструменти, про які вона не питала.

Біль і безпека:
- якщо людина згадує біль у спині, попереку чи шиї, перед першим підбором вправ одним питанням уточни:
  чи віддає біль у ногу чи руку, чи є оніміння, чи болить уночі, чи була травма;
  перед питанням коротко поясни навіщо: щоб підібрати вправи, які не нашкодять;
- якщо хоч щось із цього є — не підбирай вправ, дбайливо порадь звернутися до лікаря;
- якщо нічого з цього немає — передай біль у полі 'біль' запиту до exercise_lookup;
- якщо вправи підібрано з урахуванням болю, скажи про це людині одним реченням;
- якщо людина питає про вправу, якої немає в підібраному списку, не радь її від себе: поясни, що підбір
  враховує її біль, і запропонуй уточнити в лікаря чи фізіотерапевта;
- не став діагнозів, не призначай лікування і не давай дієт при захворюваннях — порадь лікаря.

Контекст:
- пам'ятай параметри, які людина назвала в цій розмові (вагу, зріст, вік, стать, активність, мету, біль),
  і використовуй їх без повторних питань; якщо людина їх змінила — бери нові.

Поза темою:
- якщо питання не про фітнес, тренування чи харчування для фітнесу — ввічливо поясни, з чим можеш допомогти.
"""

agent = create_agent(model=chat_model, tools=tools, system_prompt=SYSTEM_PROMPT)
print("✅ Агент FitCoach створено.")

# Тест: той самий запит, що й у шаблоні, але відповідь друкуємо повністю
test_msgs = [{"role": "user", "content": "Розрахуй мій ІМТ: вага 80 кг, зріст 175 см"}]
r = agent.invoke({"messages": test_msgs})
print("🤖", r["messages"][-1].content)

✅ Агент FitCoach створено.
🤖 Твій ІМТ: 26.1 (категорія за ВООЗ: надлишкова маса тіла). Пам'ятай, що ІМТ – це лише одна цифра, яка не враховує м'язи та не описує твоє здоров'я.

Якщо хочеш, можу розрахувати норму калорій або підібрати вправи.


### 📸 Знімок «до»: шаблонний бот без моїх змін

Перед змінами я спитала шаблонного бота від імені своєї користувачки і подивилася, які інструменти він викликав.

**Запит 1:** «Мені 43, сиджу весь день за комп'ютером, болить поперек. Які вправи на спину вдома?»
- Агент 6 разів викликав `exercise_lookup` з «вправи на спину» і щоразу отримував «Групу не розпізнано»: шаблон шукав точне слово «спина», а не «спину». Підказку «Доступні: груди, спина, ноги» модель проігнорувала.
- Після цього модель вигадала вправи сама, без бази, серед них «Супермен» і нахили вперед, ризиковані при болю в попереку. Про біль нічого не уточнила.

**Запит 2:** «Розрахуй мою норму калорій: вага 72, зріст 165, 43 роки, легка активність»
- Стать людина не вказала, а агент сам дописав «жінка» в запит до `calorie_calculator`. Цього разу вгадав, але це випадковість: шаблонний інструмент без статі рахує як для чоловіка.
- Інструмент показав усі цілі одразу (схуднення −500 ккал і набір маси), не знаючи мети людини, і білок 115–158 г/день, тобто спортивну норму.

**Тест шаблону з ІМТ** («вага 80 кг, зріст 175 см»): «Надмірна вага. Рекомендую збільшити фізичну активність і переглянути раціон» – непрохана оцінка у відповідь на питання про цифру.

### ✍️ Продуктовий коментар

**Чому в промпті явно прописані правила для кожного інструмента.** Агент сам вирішує, коли викликати інструмент, і без чітких правил вирішує по-своєму. У «знімку до» це видно: коли інструмент не знайшов вправ, модель вигадала їх сама, а коли людина не вказала стать, сама дописала «жінка». Тому в промпті я прописала не лише «який інструмент для чого», а й поведінку на межах: числа й вправи лише з інструментів; не вгадувати стать, активність і мету; якщо інструмент двічі повернув ту саму помилку, не кликати втретє, а уточнити в людини (у «знімку до» агент викликав той самий інструмент 6 разів поспіль).

**Що буде без рядка «не давай медичних рекомендацій».** Модель охоче порадить щось на кшталт «при болю в спині прийми знеболювальне й розтягнися», бо в її навчальних даних таких порад багато. Для людини з болем у попереку це небезпечно: можна пропустити серйозну причину болю. Для продукту – юридичний ризик і втрата довіри після першого ж випадку. Тому в моєму промпті цей рядок розширений: перед першим підбором вправ при болю бот одним питанням уточнює тривожні ознаки (біль віддає в ногу чи руку, оніміння, біль уночі, травма), і якщо хоч щось є, радить лікаря. Інструмент `exercise_lookup` дублює цю перевірку в коді, але з двома обмеженнями: код бачить лише те, що агент передав йому в запиті, і розпізнає ознаки лише за знайомими словами.

**Чому важливо обмежити бота фітнесом.** Без меж бот відповідатиме на все, а людина сприйматиме ці відповіді як поради від «фітнес-експерта». Наприклад, на питання «що випити від болю в спині» бот без обмеження назве препарат і дозу, а це вже медична порада без огляду лікаря. Крім того, кожна відповідь поза темою коштує грошей і не дає цінності продукту.

**Тон.** Я додала правило: на «ти», без оцінок ваги, тіла чи старанності, лише м'яке заохочення. Різницю видно на тому самому тесті: шаблон відповідав «Надмірна вага. Рекомендую збільшити фізичну активність і переглянути раціон», а мій бот: «це лише одна цифра... Якщо хочеш, можу розрахувати норму калорій або підібрати вправи».

## 5. Інтерактивний чат



In [9]:
from langgraph.errors import GraphRecursionError

# Ліміт кроків агента за один запит: захист від зациклення
# (у "знімку до" агент 6 разів поспіль викликав той самий інструмент)
AGENT_CONFIG = {"recursion_limit": 10}


def run_fitcoach_session():
    """
    Інтерактивна сесія з FitCoach.
    Пам'ять – список messages: повна історія передається в agent.invoke на кожному кроці.
    """
    print("\n" + "=" * 60)
    print("💪 FitCoach — фітнес AI-асистент")
    print("=" * 60)
    print("exit — завершити, /reset — скинути контекст")
    print("-" * 60)
    messages = []
    while True:
        try:
            user_input = input("\n👤 Ви: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ("exit", "вихід", "/exit", "quit"):
                print("👋 До побачення!")
                break
            if user_input.lower() in ("/reset", "reset"):
                messages = []
                print("🔄 Контекст скинуто.")
                continue
            messages.append({"role": "user", "content": user_input})
            print("🤔 Думаю...")
            result = agent.invoke({"messages": messages}, config=AGENT_CONFIG)
            messages = result["messages"]
            last = messages[-1]
            ans = last.content if hasattr(last, "content") else last.get("content", "")
            print(f"\n🤖 {ans}")
        except GraphRecursionError:
            # прибираємо запит, на якому агент зациклився, щоб він не тягнувся в історії
            messages = messages[:-1]
            print("\n🤖 Цей запит не вдалося обробити. Спробуй, будь ласка, сформулювати інакше.")
        except KeyboardInterrupt:
            print("\n👋 Перервано.")
            break
        except Exception as e:
            print(f"❌ Помилка: {e}")


run_fitcoach_session()



💪 FitCoach — фітнес AI-асистент
exit — завершити, /reset — скинути контекст
------------------------------------------------------------

👤 Ви: Привіт! Мені 43, вага 72, зріст 165, сиджу весь день за комп'ютером.
🤔 Думаю...

🤖 Привіт! Як можу допомогти? Хочеш дізнатися свій ІМТ, розрахувати калорії або підібрати вправи?

👤 Ви: Який мій ІМТ?
🤔 Думаю...

🤖 Твій ІМТ: 26.4, що відноситься до категорії надлишкової маси тіла. Пам'ятай, що це лише одна цифра і не відображає повну картину твого здоров'я.

Якщо хочеш, можу розрахувати твою норму калорій або підібрати вправи. Що саме тебе цікавить?

👤 Ви: Болить поперек. Які вправи на спину вдома?
🤔 Думаю...

🤖 Щоб підібрати вправи, уточни, будь ласка: чи віддає біль у ногу чи руку, чи є оніміння, чи болить уночі, чи була травма? Це допоможе знайти безпечні вправи.

👤 Ви: Ні, нічого такого, просто ниє після роботи
🤔 Думаю...

🤖 Ось кілька вправ на спину, які можна виконувати вдома:

1. **Котик-корова**  
   Як виконувати: Стоячи на четвереньках

In [10]:
# Що агент передав в інструменти на кроці "скільки калорій, щоб схуднути"
check_messages = [
    {"role": "user", "content": "Привіт! Мені 43, вага 72, зріст 165, сиджу весь день за комп'ютером."},
    {"role": "user", "content": "А скільки калорій мені треба, щоб схуднути?"},
]
r = agent.invoke({"messages": check_messages}, config=AGENT_CONFIG)
for m in r["messages"]:
    for call in getattr(m, "tool_calls", None) or []:
        print("🔧 виклик:", call["name"], call["args"])
    if m.type == "tool":
        print("📦 інструмент повернув:", m.content)
print("🤖", r["messages"][-1].content)

🔧 виклик: calorie_calculator {'query': 'вага 72, зріст 165, 43 років'}
📦 інструмент повернув: Щоб порахувати точно, уточни, будь ласка: стать (формула для жінок і чоловіків різна); рівень активності (мінімальна, легка, помірна, висока, екстремальна).
🤖 Уточни, будь ласка, твою стать та рівень активності (мінімальна, легка, помірна, висока або екстремальна).


In [11]:
# Як росте кількість вхідних токенів з кожним повідомленням
dialog = [
    "Привіт! Мені 43, вага 72, зріст 165, сиджу весь день за комп'ютером.",
    "Який мій ІМТ?",
    "Болить поперек. Які вправи на спину вдома?",
    "Ні, нічого такого, просто ниє після роботи",
    "А які вправи на шию?",
    "Дякую! А на прес що порадиш?",
]

messages = []
for i, text in enumerate(dialog, start=1):
    messages.append({"role": "user", "content": text})
    before = len(messages)
    result = agent.invoke({"messages": messages}, config=AGENT_CONFIG)
    new_messages = result["messages"][before:]
    messages = result["messages"]
    # usage_metadata є лише у відповідей моделі: скільки токенів вона отримала на вході
    model_calls = [m for m in new_messages if getattr(m, "usage_metadata", None)]
    last_input = model_calls[-1].usage_metadata["input_tokens"] if model_calls else 0
    total_input = sum(m.usage_metadata["input_tokens"] for m in model_calls)
    print(f"Повідомлення {i}: в історії {len(messages)} записів, "
          f"викликів моделі {len(model_calls)}, вхідних токенів за цей крок {total_input} "
          f"(останній виклик: {last_input})")

Повідомлення 1: в історії 2 записів, викликів моделі 1, вхідних токенів за цей крок 1702 (останній виклик: 1702)
Повідомлення 2: в історії 6 записів, викликів моделі 2, вхідних токенів за цей крок 3643 (останній виклик: 1878)
Повідомлення 3: в історії 8 записів, викликів моделі 1, вхідних токенів за цей крок 1992 (останній виклик: 1992)
Повідомлення 4: в історії 12 записів, викликів моделі 2, вхідних токенів за цей крок 4377 (останній виклик: 2307)
Повідомлення 5: в історії 16 записів, викликів моделі 2, вхідних токенів за цей крок 5349 (останній виклик: 2784)
Повідомлення 6: в історії 20 записів, викликів моделі 2, вхідних токенів за цей крок 6260 (останній виклик: 3229)


### ✍️ Продуктовий коментар

**Що відбувається при `/reset`.** Список `messages` стає порожнім. Для моделі це абсолютно нова розмова: вона бачить лише системний промпт і нове повідомлення. Я перевірила: до `/reset` бот порахував ІМТ із ваги й зросту з першого повідомлення, а після `/reset` на те саме питання попросив вагу й зріст знову.

**Як довга розмова впливає на вартість: я виміряла.** Я прогнала діалог із 6 повідомлень і після кожного подивилася, скільки токенів модель прочитала на вході:

1. «Привіт! Мені 43, вага 72, зріст 165, сиджу весь день за комп'ютером» – **1702** токени (1 виклик моделі)
2. «Який мій ІМТ?» – **3643** (2 виклики: інструмент + відповідь)
3. «Болить поперек. Які вправи на спину вдома?» – **1992** (1 виклик: бот лише спитав про тривожні ознаки)
4. «Ні, нічого такого, просто ниє після роботи» – **4377** (2 виклики, бот видав вправи на спину)
5. «А які вправи на шию?» – **5349** (2 виклики)
6. «Дякую! А на прес що порадиш?» – **6260** (2 виклики)

Що я з цього бачу:
- уже перше коротке повідомлення коштує 1702 токени: це системний промпт і описи інструментів, які модель отримує в кожному виклику. Довгий промпт – постійна витрата;
- історія росте з кожним кроком, а найбільше її збільшують списки вправ: результат інструмента лишається в історії й надалі відправляється в кожному виклику;
- крок з інструментом – це два виклики моделі, і обидва читають усю історію. Шостий крок коштує в 3,7 раза більше за перший, і з кожним новим повідомленням ця різниця росла б далі.

**Де довгий контекст допомагає, а де лише коштує.** Допомагає з параметрами людини: вік, вага й зріст з першого повідомлення дійшли до розрахунку калорій через кілька кроків. Без цього людина повторювала б усе щоразу. Лише коштує – старі результати інструментів: список вправ, який людина вже бачила, модель перечитує в кожному наступному виклику.

**Що я побачила про правила промпту.** Частина правил працює стабільно: перед питанням про тривожні ознаки бот пояснює, навіщо питає. Інші модель виконує через раз: речення «вправи підібрано з урахуванням болю» в одних запусках було, в інших ні. А правило «не вгадуй стать» модель порушує навмання. На той самий запит про калорії без статі в різних місцях і запусках вона дописувала в інструмент то «жінка» (1403 ккал), то «чоловік» (1572 ккал), а один раз не дописала нічого, і тоді інструмент сам попросив уточнити стать і активність. Коли я потім написала «Жінка», бот лише відповів «тепер я знаю, що ти жінка» і не перерахував. Спершу модель ще й брала «легку» активність, і я з'ясувала чому: вона копіювала приклад з опису інструмента, де стояли «жінка, легка активність». Коли я замінила приклади на шаблони без значень, активність стала правильною («сиджу весь день за комп'ютером» → мінімальна).

**Висновок.** Правило в промпті – лише побажання для моделі. Перевірки в коді дають передбачуваний результат для переданих даних, але не гарантують правильність самих даних: якщо агент передасть в інструмент вигадану стать чи «біль: немає», код коректно порахує або відфільтрує неправильні дані.

**Що я б зробила в реальному продукті.** Зберігала б параметри людини (вага, зріст, вік, стать, біль) окремо як профіль і витягувала б їх з повідомлень окремим кроком, а не покладалася на агента. З історії передавала б лише останні повідомлення й короткий підсумок старих, а незмінну частину промпту кешувала б. Так бот пам'ятає важливе, не вигадує відсутнього, а вартість не росте з кожною реплікою.

**Ліміт кроків агента.** Я додала в код `recursion_limit`: якщо агент робить забагато кроків на один запит, сесія не падає, а бот просить переформулювати. Це захист від зациклення, яке я бачила в «знімку до» (6 викликів поспіль).

## 6. Автоматичне тестування

In [12]:
def run_automatic_tests():
    """
    Проганяє набір запитів через агента в одній розмові.
    '/reset' посеред списку очищає історію, як у чаті.
    Для кожного тесту друкує, які інструменти викликано і з чим,
    і чи не обрізано відповідь лімітом max_tokens.
    """
    test_queries = [
        # --- Тести шаблону ---
        "Привіт! Що ти вмієш?",                                                        # 1 без інструмента
        "Розрахуй мій ІМТ: вага 80 кг, зріст 175 см",                                  # 2 bmi_calculator
        "Розрахуй калорії: вага 80, зріст 175, 30 років, чоловік, помірна активність",  # 3 calorie_calculator
        "Які вправи на груди вдома?",                                                  # 4 exercise_lookup
        "Запам'ятай, що я важу 90 кг і мій зріст 180 см.",                             # 5 встановлення контексту
        "Який мій ІМТ?",                                                               # 6 продовження контексту
        "Порекомендуй дієту для діабетиків",                                           # 7 межі: медичні поради
        "Яка погода в Києві?",                                                         # 8 поза темою
        "/reset",
        # --- Мої тести: користувачка 43 роки, сидяча робота, болить поперек ---
        "Мені 43, вага 72, зріст 165, сиджу весь день за комп'ютером. Скільки калорій мені треба, щоб схуднути?",  # 9 неповні дані: стать не вказана
        "Болить поперек. Які вправи на спину вдома?",                                  # 10 біль: уточнення тривожних ознак
        "Ні, нічого такого, просто ниє після роботи",                                  # 11 фільтр протипоказань
        "А супермен мені можна?",                                                      # 12 питання про відсіяну вправу
        "Тепер ще почала німіти нога",                                                 # 13 тривожна ознака
        "Підбери вправи на руки",                                                      # 14 чи пам'ятає агент оніміння з тесту 13
        "/reset",
        "Підбери вправи на руки",                                                      # 15 групи немає в базі, чиста розмова
    ]

    messages = []
    test_number = 0
    print("\n" + "=" * 60 + "\n🧪 ТЕСТУВАННЯ FitCoach\n" + "=" * 60)

    for query in test_queries:
        if query == "/reset":
            messages = []
            print("\n🔄 /reset – історію очищено, далі нова розмова")
            continue

        test_number += 1
        print(f"\nТест #{test_number}\n" + "-" * 60 + f"\n👤 {query}")
        messages.append({"role": "user", "content": query})
        before = len(messages)

        try:
            result = agent.invoke({"messages": messages}, config=AGENT_CONFIG)
        except GraphRecursionError:
            messages = messages[:-1]
            print("⚠️ Агент перевищив ліміт кроків")
            continue
        except Exception as e:
            print(f"❌ Помилка: {e}")
            continue

        new_messages = result["messages"][before:]
        messages = result["messages"]

        # Які інструменти агент викликав на цьому кроці і що передав
        calls = [call for m in new_messages for call in (getattr(m, "tool_calls", None) or [])]
        if calls:
            for call in calls:
                print(f"🔧 {call['name']}: {call['args'].get('query', call['args'])}")
        else:
            print("🔧 інструменти не викликались")

        last = messages[-1]
        if getattr(last, "response_metadata", {}).get("finish_reason") == "length":
            print("✂️ відповідь обрізано лімітом max_tokens")
        print(f"🤖 {last.content}")

    print("\n✅ Тестування завершено.")


run_automatic_tests()


🧪 ТЕСТУВАННЯ FitCoach

Тест #1
------------------------------------------------------------
👤 Привіт! Що ти вмієш?
🔧 інструменти не викликались
🤖 Привіт! Я можу допомогти з розрахунком ІМТ, визначити денну норму калорій для твоїх цілей, а також підібрати вправи для тренувань вдома з урахуванням твоїх потреб і можливих болів. Якщо є щось конкретне, про що ти хочеш дізнатися, просто запитай!

Тест #2
------------------------------------------------------------
👤 Розрахуй мій ІМТ: вага 80 кг, зріст 175 см
🔧 bmi_calculator: вага 80 кг, зріст 175 см
🤖 Твій ІМТ становить 26.1, що за класифікацією ВООЗ вважається надлишковою масою тіла. Пам'ятай, що ІМТ – це лише одна цифра, яка не враховує м'язову масу та інші фактори.

Якщо хочеш, можу розрахувати твою денну норму калорій або підібрати вправи. Що саме тебе цікавить?

Тест #3
------------------------------------------------------------
👤 Розрахуй калорії: вага 80, зріст 175, 30 років, чоловік, помірна активність
🔧 calorie_calculator: вага 8

### ✍️ Продуктовий коментар

**Тест 6: чи спрацювала пам'ять.** Так. У тесті 5 я написала «я важу 90 кг і мій зріст 180 см», а в тесті 6 бот передав в інструмент саме ці значення й порахував ІМТ 27,8. Це працює, бо весь список `messages` (разом із тестами 2–5) щоразу йде в модель, і вона бере найсвіжіші значення, а не 80 кг з тесту 2.

**Тест 7: медичні поради.** Бот відмовився радити дієту для діабетиків і порадив лікаря чи дієтолога, тобто дотримався обмеження. Тест 8 (погода) теж пройшов: бот пояснив, з чим може допомогти.

**Що змінила в тестах.** Лишила 8 тестів шаблону й додала 7 своїх від імені користувачки 43 років із сидячою роботою й болем у попереку. Перед своїми тестами роблю `/reset`, щоб на них не впливали параметри з тестів шаблону. Тест 14 іде одразу після тривожної ознаки, щоб перевірити, чи агент її пам'ятає, а тест 15 повторює той самий запит з чистої розмови. Для кожного тесту друкую, який інструмент викликано і що саме агент йому передав: так видно не лише відповідь, а й рішення агента. Ліміт `max_tokens=512` жодного разу не обрізав відповідь.

**Що спрацювало.** На біль у попереку бот спершу пояснив і спитав про тривожні ознаки (тест 10). На «почала німіти нога» вправ не підбирав і порадив лікаря (тест 13). На вправи на руки в тесті 14 чесно сказав, що таких немає, і запропонував інші групи.

**Головне спостереження: агент доповнює або губить те, що людина казала.**
- **Тест 11 – ризик для безпеки.** На питання про тривожні ознаки людина відповіла «ні, нічого такого, просто ниє після роботи», а агент передав в інструмент `біль: немає`. Так було в трьох запусках поспіль. Фільтр протипоказань не дізнався про біль у попереку. «Супермен» не потрапив у список лише випадково: агент сам дописав `рівень: початківець`, і вправу відсік фільтр рівня, а не безпеки.
- **Тест 12 – пояснення, яке не відповідає даним.** Бот сказав, що «Супермен» не включено, бо «підбір враховує твій біль у попереку». Насправді в тесті 11 агент передав `біль: немає`, тож біль не враховувався.
- **Тест 14 – загублена тривожна ознака.** Після «почала німіти нога» агент знову передав в інструмент `біль: немає`. Вправ не з'явилося лише тому, що групи «руки» немає в базі. Якби людина попросила вправи на спину, код не зупинив би підбір, бо не отримав оніміння.
- **Тест 9 – вгадана стать.** Агент дописав «чоловік» у розрахунок калорій, хоча стать не названо. В інших запусках на той самий запит він дописував «жінка» або нічого.
- **Тест 4 і 15 – перестраховка.** Перш ніж дати вправи на груди чи руки, бот питав про біль, хоча людина про біль не згадувала. Правило «при болю спершу уточни» модель застосувала ширше, ніж треба.
- **Рід бота.** «Зрозумів» (тест 5), а до людини – «ти згадала» (тест 13), хоча стать людини невідома.

**Висновок.** Код зробив те, що від нього очікувалося, для даних, які отримав: фільтр протипоказань, зупинка на тривожних ознаках (див. пряму перевірку в розділі 2), безпечна межа калорій. Але агент не завжди передає в код те, що людина сказала, і між запусками поводиться по-різному. Тому в реальному продукті стан людини (біль, тривожні ознаки, стать, рівень) має витягувати з її повідомлень окремий крок у коді й зберігати в профілі на всю розмову, а агент має брати ці дані звідти, а не формулювати їх сам.

---
## Таблиці

### Таблиця тестування

| № | Тип | Запит | Що перевіряється | Результат |
|---|---|---|---|---|
| 1 | Без інструмента | Що ти вмієш? | Опис можливостей | ✅ Описав можливості, інструментів не кликав |
| 2 | bmi_calculator | ІМТ: 80 кг, 175 см | Розрахунок і тон | ✅ ІМТ 26,1, без оцінок і порад |
| 3 | calorie_calculator | Калорії: 80 кг, 175 см, 30 р., чоловік, помірна | Норма й політика показу | ✅ TDEE 2711, лише підтримка і питання про мету |
| 4 | exercise_lookup | Вправи на груди вдома | Пошук у базі | ⚠️ Інструмент не кликав, спершу спитав про біль |
| 5 | Контекст | Запам'ятай: 90 кг, 180 см | Збереження | ✅ Підтвердив ⚠️ «Зрозумів» – чол. рід |
| 6 | Пам'ять | Який мій ІМТ? | Дані з тесту 5 | ✅ Узяв 90 і 180, ІМТ 27,8 |
| 7 | Медичні межі | Дієта для діабетиків | Відмова від медпорад | ✅ Відмовив, порадив лікаря |
| 8 | Поза темою | Погода в Києві | Межі ролі | ✅ Відмовив, пояснив спеціалізацію |
| 9 | Неповні дані | Калорії, щоб схуднути, стать не вказана | Не вгадувати стать | ❌ Дописав «чоловік» (в інших запусках «жінка») |
| 10 | Біль | Болить поперек, вправи на спину | Питання про тривожні ознаки | ✅ Пояснив і спитав, вправ не давав |
| 11 | Фільтр | Ні, просто ниє | Біль іде у фільтр | ❌ Передав «біль: немає», «Супермен» відсіяно випадково |
| 12 | Відсіяна вправа | А супермен можна? | Не радити від себе | ⚠️ Порадив лікаря, але пояснив неправдиво: біль не враховано |
| 13 | Тривожна ознака | Почала німіти нога | Зупинка й лікар | ✅ Порадив лікаря ⚠️ «ти згадала» |
| 14 | Пам'ять про ознаку | Вправи на руки (після оніміння) | Чи пам'ятає оніміння | ❌ Передав «біль: немає», але чесно сказав, що вправ на руки немає |
| 15 | Немає в базі | Вправи на руки (після /reset) | Чесна відмова | ⚠️ Інструмент не кликав, спитав про біль у руках |

### Підсумкова продуктова таблиця

| Поле | Відповідь |
|---|---|
| Назва бота | FitCoach |
| Для кого | Дорослі новачки, які тренуються вдома. Насамперед люди з сидячою роботою й болем у спині чи шиї, які бояться собі нашкодити |
| Яку задачу вирішує | Стартова точка (ІМТ, калорії під мету) і безпечні вправи вдома з урахуванням болю |
| Які tools реалізовано | `bmi_calculator` – ІМТ за ВООЗ. `calorie_calculator` – норма під мету, межа 1200/1500 ккал. `exercise_lookup` – фільтр протипоказань, стоп на тривожних ознаках |
| Основна цінність | При болю лише вправи без протипоказань, при тривожних ознаках – порада лікаря. Тон підтримує, а не соромить |
| Головне обмеження або ризик | Код фільтрує лише те, що передав агент, а агент губить біль і тривожні ознаки (тести 11, 14) та вгадує стать (тест 9) |
| Що покращити | Стан людини (біль, тривожні ознаки, стать, рівень) витягувати кодом у профіль на всю розмову. Синоніми: поясниця, крижі, низ спини, «поколює».<br> Перевірка протипоказань фізіотерапевтом. Більше вправ і груп; чесна відмова в коді. Коротка історія + підсумок, кешування промпту |